# 04 — Classical baseline (Stage C, Arm 1)

Logistic regression and gradient boosting on the ten hand-crafted quality features, one model per
output (BLR BRT DRK FRM OBS ROT at >=2 votes, plus UNREC = unrecognizable >=2), at two feature
resolutions (long edge 224 and 512). All modelling logic lives in `src/models/`; this notebook only
calls it and displays results. It regenerates `reports/results_classical.md` and
`reports/results_classical.csv`.

**Partitions:** models are fitted on train and scored on val. The test partition is never read.

**How to run:** from the `notebooks/` folder, with `DATA_DIR` pointing at the data directory
(defaults to `../data`). `DATA_DIR` must contain `annotations/{train,val}.json`,
`classical_features_{224,512}.csv` (from `python scripts/extract_classical_features.py DATA_DIR`)
and `image_properties.csv` (from `scripts/extract_image_properties.py`). No GPU needed; the whole
notebook runs in a few minutes on a CPU.

In [2]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
DATA_DIR = Path(os.environ.get("DATA_DIR", REPO_DIR / "data"))
REPORT_PATH = REPO_DIR / "reports" / "results_classical.md"
RESULTS_CSV = REPO_DIR / "reports" / "results_classical.csv"
PREDICTIONS_DIR = DATA_DIR / "predictions"

from src.data.splits import MODELLED_FLAWS, SEED, get_splits
from src.eda import image_props as ip
from src.eda import labels as lb
from src.eda import report as rp
from src.features.classical import FEATURE_NAMES
from src.models import classical as cm
from src.models import metrics as mt

random.seed(SEED)
np.random.seed(SEED)
RESOLUTIONS = (224, 512)
ENVIRONMENT = rp.environment_lines(
    gpu_note="not needed: scikit-learn models on precomputed features",
    seed_note="gradient boosting random_state; logistic regression is deterministic")
pd.set_option("display.precision", 3)

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)
print("\n".join(ENVIRONMENT))

REPO_DIR: /content/drive/MyDrive/Masters/Sem 3/capstone/repo
DATA_DIR: /content/drive/MyDrive/Masters/Sem 3/capstone/data
- Python 3.13.16 on Linux-6.6.122+-x86_64-with-glibc2.39
- RAM: 12.7 GB
- GPU: none (not needed: scikit-learn models on precomputed features)
- Seed: 42 (gradient boosting random_state; logistic regression is deterministic)


## 1. Data
Targets come from the annotation records via `src/data/splits.py`; features are joined by image
name. Every train and val image must have a feature row at both resolutions.

In [3]:
splits = get_splits(DATA_DIR)
targets = {p: cm.build_targets(splits[p]) for p in ("train", "val")}
features = {res: cm.load_features(DATA_DIR, res) for res in RESOLUTIONS}
for res, df in features.items():
    for p, t in targets.items():
        missing = t.index.difference(df.index)
        assert missing.empty, f"{len(missing)} {p} images missing from the {res}px features"

X = {(res, p): features[res].loc[targets[p].index] for res in RESOLUTIONS for p in targets}
sections = []  # (heading, markdown) pairs collected for the report

positives = pd.DataFrame({p: t.sum() for p, t in targets.items()})
positives["val %"] = (positives["val"] / len(targets["val"]) * 100).round(1)
print({p: len(t) for p, t in targets.items()})
display(positives)
sections.append(("Data", (
    f"Train {len(targets['train']):,} images (fitting), val {len(targets['val']):,} images (scoring). "
    "Positives per output at >=2 votes:\n\n" + rp.df_to_markdown(positives))))

{'train': 23431, 'val': 3875}


,train,val,val %
BLR,9552,1636,42.2
BRT,1248,224,5.8
DRK,1367,244,6.3
FRM,12934,2148,55.4
OBS,840,137,3.5
ROT,3986,664,17.1
UNREC,3558,606,15.6


## 2. Feature extraction summary
Counts are recomputed from the feature CSVs. The run details (runtime, verification) are recorded
from the extraction run itself.

In [4]:
uniform = features[224]["lum_std"] == 0
extraction = pd.DataFrame({
    f"{res}px": {"rows": len(df), "upsampled (native long edge < target)": int(df["upsampled"].sum()),
                 "upsampled %": round(df["upsampled"].mean() * 100, 1)}
    for res, df in features.items()})
display(extraction)
print("fully uniform images (lum_std = 0):", int(uniform.sum()))
sections.append(("Feature extraction", (
    "Features from `scripts/extract_classical_features.py` on the original images (PIL, no EXIF "
    "rotation), resized to a fixed long edge with aspect preserved and no padding. One code path, "
    "both resolutions.\n\n" + rp.df_to_markdown(extraction) + "\n\n"
    f"- {int(uniform.sum())} images are fully uniform (luminance std 0).\n"
    "- Run: 2026-10-10 on a Colab CPU runtime, single process, about 50 minutes for 27,306 images at "
    "both resolutions (~560 images/min).\n"
    "- Verification (runner check, 26/26 passed): exact train+val image set with no duplicates and "
    "no test images; native sizes equal `image_properties.csv` (no EXIF rotation); upsampled flag "
    "consistent; all values finite and in range; five random images recomputed from the original "
    "zips match their rows exactly.")))

,224px,512px
rows,27306.0,27306.0
upsampled (native long edge < target),92.0,2546.0
upsampled %,0.3,9.3


fully uniform images (lum_std = 0): 19


## 3. Fully uniform images
Images with zero luminance variation are blank frames (black, white or flat). They should be
labelled mostly DRK, BRT or UNREC; if they are not, the labels and features disagree.

In [5]:
labelled = pd.concat([targets["train"], targets["val"]])
u = features[224].loc[uniform, ["lum_mean"]].join(labelled)
display(u.sort_values("lum_mean"))
uniform_rates = (u[cm.OUTPUTS].mean() * 100).round(0).to_frame("% of uniform images positive").T
display(uniform_rates)
sections.append(("Fully uniform images", (
    f"{len(u)} images with luminance std 0. Share positive per output:\n\n"
    + rp.df_to_markdown(uniform_rates, index=False))))

,lum_mean,BLR,BRT,DRK,FRM,OBS,ROT,UNREC
image,,,,,,,,
VizWiz_train_00022614.jpg,0.0,0,0,1,0,0,0,1
VizWiz_train_00000611.jpg,254.0,0,1,0,0,0,0,1
VizWiz_train_00007901.jpg,254.0,0,1,0,0,0,0,1
VizWiz_train_00000635.jpg,255.0,0,0,1,0,0,0,1
VizWiz_train_00001734.jpg,255.0,0,1,0,0,0,0,1
VizWiz_train_00004302.jpg,255.0,0,0,0,0,0,0,1
VizWiz_train_00005104.jpg,255.0,0,0,0,0,0,0,1
VizWiz_train_00000396.jpg,255.0,0,0,0,0,0,0,1
VizWiz_train_00009464.jpg,255.0,0,0,0,0,0,0,1


,BLR,BRT,DRK,FRM,OBS,ROT,UNREC
% of uniform images positive,0.0,37.0,11.0,0.0,0.0,0.0,100.0


## 4. Train
Four configurations (2 resolutions x 2 model kinds), seven models each. Every fitted component,
scaler included, sees train only; val is only scored. Val scores are saved to
`DATA_DIR/predictions/` (data, not committed) for the Stage F comparison.

In [6]:
models, scores = {}, {}
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
for res in RESOLUTIONS:
    for kind in cm.KINDS:
        print(f"{kind} @ {res}px")
        models[res, kind] = cm.train_models(X[res, "train"], targets["train"], kind)
        scores[res, kind] = cm.predict_scores(models[res, kind], X[res, "val"])
        scores[res, kind].to_csv(PREDICTIONS_DIR / f"classical_{kind}_{res}_val.csv")
print("val scores saved to", PREDICTIONS_DIR)

lr @ 224px
  [lr] BLR     9552 positives  fitted in   0.1s
  [lr] BRT     1248 positives  fitted in   0.1s
  [lr] DRK     1367 positives  fitted in   0.1s
  [lr] FRM    12934 positives  fitted in   0.1s
  [lr] OBS      840 positives  fitted in   0.1s
  [lr] ROT     3986 positives  fitted in   0.0s
  [lr] UNREC   3558 positives  fitted in   0.1s
gb @ 224px
  [gb] BLR     9552 positives  fitted in   0.3s
  [gb] BRT     1248 positives  fitted in   0.2s
  [gb] DRK     1367 positives  fitted in   0.2s
  [gb] FRM    12934 positives  fitted in   0.3s
  [gb] OBS      840 positives  fitted in   0.1s
  [gb] ROT     3986 positives  fitted in   0.2s
  [gb] UNREC   3558 positives  fitted in   0.3s
lr @ 512px
  [lr] BLR     9552 positives  fitted in   0.1s
  [lr] BRT     1248 positives  fitted in   0.1s
  [lr] DRK     1367 positives  fitted in   0.1s
  [lr] FRM    12934 positives  fitted in   0.1s
  [lr] OBS      840 positives  fitted in   0.1s
  [lr] ROT     3986 positives  fitted in   0.0s
  [lr] 

## 5. Results on val
AP is the primary metric and is threshold-free; **read it against the base rate** (the AP of a
random scorer). Precision, recall and F1 are at a fixed 0.5 threshold. The macro row averages the
six flaws; UNREC is the separate recognisability head.

In [7]:
results = {cfg: mt.evaluate(targets["val"], s) for cfg, s in scores.items()}
for (res, kind), table in results.items():
    print(f"\n{kind} @ {res}px")
    display(table)

def config_name(cfg):
    res, kind = cfg
    return f"{kind.upper()} @ {res}"

base = results[224, "lr"]["base rate"]
ap = pd.DataFrame({config_name(c): t["AP"] for c, t in results.items()})
ap.insert(0, "base rate", base)
f1 = pd.DataFrame({config_name(c): t["F1"] for c, t in results.items()})
print("\nAP by configuration")
display(ap)
print("F1 @ 0.5 by configuration")
display(f1)

long = pd.concat({config_name(c): t for c, t in results.items()}, names=["config", "output"])
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
long.reset_index().to_csv(RESULTS_CSV, index=False, float_format="%.4f")

sections.append(("Results on val: AP", (
    "Average precision per output. The base rate is the AP of a random scorer.\n\n"
    + rp.df_to_markdown(ap.round(3)))))
sections.append(("Results on val: F1 at threshold 0.5", rp.df_to_markdown(f1.round(3))))
for cfg, table in results.items():
    sections.append((f"Full table: {config_name(cfg)}", rp.df_to_markdown(table.round(3))))


lr @ 224px


,n,positives,base rate,AP,precision,recall,F1
BLR,3875,1636,0.422,0.798,0.740,0.679,0.708
BRT,3875,224,0.058,0.428,0.189,0.732,0.301
DRK,3875,244,0.063,0.556,0.220,0.770,0.342
FRM,3875,2148,0.554,0.679,0.644,0.613,0.628
OBS,3875,137,0.035,0.102,0.073,0.657,0.132
ROT,3875,664,0.171,0.219,0.213,0.645,0.320
UNREC,3875,606,0.156,0.731,0.467,0.776,0.583
macro (6 flaws),<NA>,<NA>,0.217,0.464,0.346,0.683,0.405



gb @ 224px


,n,positives,base rate,AP,precision,recall,F1
BLR,3875,1636,0.422,0.809,0.743,0.714,0.728
BRT,3875,224,0.058,0.472,0.235,0.754,0.359
DRK,3875,244,0.063,0.551,0.242,0.717,0.362
FRM,3875,2148,0.554,0.698,0.672,0.645,0.658
OBS,3875,137,0.035,0.126,0.084,0.577,0.146
ROT,3875,664,0.171,0.238,0.222,0.634,0.329
UNREC,3875,606,0.156,0.730,0.509,0.762,0.610
macro (6 flaws),<NA>,<NA>,0.217,0.482,0.366,0.674,0.430



lr @ 512px


...

,n,positives,base rate,AP,precision,recall,F1
BLR,3875,1636,0.422,0.774,0.694,0.737,0.715
BRT,3875,224,0.058,0.451,0.220,0.696,0.334
DRK,3875,244,0.063,0.541,0.250,0.717,0.371
FRM,3875,2148,0.554,0.698,0.672,0.660,0.666
OBS,3875,137,0.035,0.107,0.076,0.518,0.132
ROT,3875,664,0.171,0.230,0.220,0.611,0.324
UNREC,3875,606,0.156,0.729,0.516,0.749,0.611
macro (6 flaws),<NA>,<NA>,0.217,0.467,0.355,0.657,0.424



AP by configuration


,base rate,LR @ 224,GB @ 224,LR @ 512,GB @ 512
BLR,0.422,0.798,0.809,0.738,0.774
BRT,0.058,0.428,0.472,0.425,0.451
DRK,0.063,0.556,0.551,0.549,0.541
FRM,0.554,0.679,0.698,0.663,0.698
OBS,0.035,0.102,0.126,0.103,0.107
ROT,0.171,0.219,0.238,0.213,0.230
UNREC,0.156,0.731,0.730,0.722,0.729
macro (6 flaws),0.217,0.464,0.482,0.449,0.467


F1 @ 0.5 by configuration


,LR @ 224,GB @ 224,LR @ 512,GB @ 512
BLR,0.708,0.728,0.698,0.715
BRT,0.301,0.359,0.308,0.334
DRK,0.342,0.362,0.337,0.371
FRM,0.628,0.658,0.639,0.666
OBS,0.132,0.146,0.129,0.132
ROT,0.320,0.329,0.311,0.324
UNREC,0.583,0.610,0.582,0.611
macro (6 flaws),0.405,0.430,0.404,0.424


## 6. ROT against the shape-only reference
Landscape-shaped raw pixels predict ROT without any image content (Stage B: AP 0.258 on train).
The same reference is computed here on val, and ROT AP is also split by EXIF orientation:
within orientation 1 the shape cue carries no signal, so that row isolates content.

In [8]:
props = ip.load_properties(DATA_DIR / "image_properties.csv")
val_props = props[props["image"].isin(targets["val"].index)]
shape_ref = ip.rot_shortcut_scores(val_props, lb.vote_matrix(splits["val"]))
display(shape_ref)

orientation = props.set_index("image").loc[targets["val"].index, "orientation"]
rot_rows = {}
for cfg, s in scores.items():
    row = {}
    for tag in (1, 6):
        idx = orientation.index[orientation == tag]
        y = targets["val"].loc[idx, "ROT"]
        row[f"AP, orientation {tag} (n={len(idx)}, base {y.mean():.3f})"] = mt.average_precision(y, s.loc[idx, "ROT"])
    rot_rows[config_name(cfg)] = row
rot_by_orientation = pd.DataFrame(rot_rows).T
display(rot_by_orientation)
sections.append(("ROT against the shape-only reference (val)", (
    "Geometry-only cues scored as ROT predictors on val:\n\n" + rp.df_to_markdown(shape_ref)
    + "\n\nClassical ROT AP by EXIF orientation:\n\n" + rp.df_to_markdown(rot_by_orientation.round(3)))))

,n,cue on %,ROT prevalence %,AP,ROT % | cue on,ROT % | cue off
landscape-shaped pixels (all images),3875.0,6.8,17.1,0.254,54.9,14.4
EXIF orientation 6 (all images),3875.0,6.1,17.1,0.267,60.9,14.3
landscape-shaped pixels (orientation 1 only),3637.0,0.7,14.3,0.143,0.0,14.4


,"AP, orientation 1 (n=3637, base 0.143)","AP, orientation 6 (n=238, base 0.609)"
LR @ 224,0.183,0.786
GB @ 224,0.197,0.747
LR @ 512,0.182,0.742
GB @ 512,0.196,0.730


## 7. BLR on natively small images
At 512px, images whose original long edge is below 512 were upsampled, so they carry no extra
high-frequency detail. BLR AP is split by native size to check the 512 configuration on them.

In [9]:
native_long = features[224].loc[targets["val"].index, ["native_width", "native_height"]].max(axis=1)
blr_rows = {}
for cfg, s in scores.items():
    row = {}
    for label, mask in (("native < 512", native_long < 512), ("native >= 512", native_long >= 512)):
        y = targets["val"].loc[mask, "BLR"]
        row[f"{label} (n={int(mask.sum())}, base {y.mean():.3f})"] = mt.average_precision(y, s.loc[y.index, "BLR"])
    blr_rows[config_name(cfg)] = row
blr_by_size = pd.DataFrame(blr_rows).T
display(blr_by_size)
sections.append(("BLR by native resolution (val)", rp.df_to_markdown(blr_by_size.round(3))))

,"native < 512 (n=361, base 0.427)","native >= 512 (n=3514, base 0.422)"
LR @ 224,0.716,0.807
GB @ 224,0.759,0.813
LR @ 512,0.624,0.752
GB @ 512,0.710,0.780


## 8. Single-flaw check
AP for each flaw on images carrying no other modelled flaw (positives: that flaw alone; negatives:
no flaw). If DRK or BRT keep their AP here, the classical score measures exposure rather than
co-occurring blur.

In [10]:
single = {config_name(c): mt.single_flaw_ap(targets["val"], s) for c, s in scores.items()}
first = next(iter(single.values()))
single_table = first[["n", "positives (flaw alone)", "base rate"]].copy()
for name, t in single.items():
    single_table[f"AP {name}"] = t["AP"]
display(single_table)
sections.append(("Single-flaw check (val)", rp.df_to_markdown(single_table.round(3))))

,n,positives (flaw alone),base rate,AP LR @ 224,AP GB @ 224,AP LR @ 512,AP GB @ 512
BLR,1254.0,402.0,0.321,0.730,0.729,0.648,0.682
BRT,876.0,24.0,0.027,0.361,0.447,0.350,0.430
DRK,910.0,58.0,0.064,0.853,0.810,0.851,0.812
FRM,1644.0,792.0,0.482,0.582,0.626,0.581,0.627
OBS,869.0,17.0,0.020,0.079,0.115,0.070,0.115
ROT,1011.0,159.0,0.157,0.181,0.194,0.177,0.194


## 9. What logistic regression relies on
Weights on standardised features (log1p for the heavy-tailed ones), 224px. Sign is the direction,
magnitude the strength. Sanity check: DRK should lean on low `lum_mean` / high `clip_dark_frac`,
BRT the opposite, BLR on low sharpness.

In [11]:
coefs = cm.lr_coefficients(models[224, "lr"])
display(coefs.style.background_gradient(cmap="RdBu_r", axis=None, vmin=-abs(coefs).max().max(),
                                        vmax=abs(coefs).max().max()).format("{:.2f}"))
sections.append(("Logistic regression weights (224px, standardised features)",
                 rp.df_to_markdown(coefs.round(2))))

,BLR,BRT,DRK,FRM,OBS,ROT,UNREC
laplacian_var,-4.84,-0.98,0.21,-0.78,0.29,-1.00,-1.02
tenengrad,0.41,1.76,-1.18,-0.75,-0.25,0.25,-1.60
lum_mean,-0.14,-0.67,-1.41,-0.08,-0.12,-0.12,-0.15
lum_std,-0.01,0.32,0.39,0.04,0.59,0.05,-0.03
clip_dark_frac,-0.22,-0.53,0.15,-0.10,-0.20,-0.08,0.10
clip_bright_frac,-0.03,1.24,0.23,-0.10,0.02,-0.17,0.36
hf_residual_energy,1.42,0.77,-0.24,-0.06,-0.46,0.77,-0.06
border_edge_energy,0.09,-0.04,-0.25,1.10,-0.12,0.21,0.64
center_edge_energy,1.93,-2.41,0.16,0.37,-0.43,0.21,0.44
border_edge_share,0.29,-0.54,-0.04,0.03,0.16,-0.14,0.14


## 10. Write the report
The findings block in `reports/results_classical.md` is hand-written after reading the results and
is preserved on every rerun.

In [12]:
out = rp.write_report(REPORT_PATH, sections, title="Stage C: Classical baseline (Arm 1)",
                      environment=ENVIRONMENT, findings_heading="Findings (hypothesis check)")
print("Report written to", out)
print("Results table written to", RESULTS_CSV)

Report written to /content/drive/MyDrive/Masters/Sem 3/capstone/repo/reports/results_classical.md
Results table written to /content/drive/MyDrive/Masters/Sem 3/capstone/repo/reports/results_classical.csv
